In [1]:
import pandas as pd
import numpy as np
import requests as rq
import sqlite3 as sql3
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from random import randint
import os

load_dotenv()

NASA_API=os.getenv('NASA_API')
URL = "https://api.nasa.gov/planetary/apod?api_key=" + str(NASA_API)


In [2]:
response = rq.get(URL)

if response.status_code == 200:
    data=response.json()
    print("Request successful")
else:
    print("An error occurred: ", response.status_code)

Request successful


In [3]:
title = data.get('title', '')
date = data.get('date', '01-01-0001')
exp = data.get('explanation', '')
img_url = data.get('url', '')
copyr = data.get('copyright', '')

In [24]:
with sql3.connect('apod_data.db') as conn:
    cursor = conn.cursor()

In [41]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS apod_images (
        id INTEGER PRIMARY KEY,
        title TEXT UNIQUE NOT NULL,
        date TEXT UNIQUE NOT NULL,
        explanation TEXT NOT NULL,
        url TEXT NOT NULL,
        copyright TEXT
    )
''')

In [42]:
try:
    cursor.execute(f'''
        INSERT INTO apod_images 
        (title, date, explanation, url, copyright)
        VALUES 
        (?, ?, ?, ?, ?)
        ''', (title, date, exp, img_url, copyr))

    conn.commit()
except Exception as e:
    print("Error: ", e)


In [43]:
cursor.execute('''
    SELECT * FROM apod_images;
''').fetchall()

[(1,
  'The Astrosphere of HD 61005',
  '2026-03-06',
  'Do young stars blow bubbles? The larger view shows a stellar field observed with the Cerro Tololo Inter-American Observatory in Chile, and the inset highlights HD 61005, a star like our Sun, only 120 light-years away. Much younger than the Sun, at just about 100 million years old, it blows a fast and dense stellar wind that pushes out the cooler dust and gas that surrounds it, forming a bubble called an astrosphere. The star-blown bubble was detected with the Chandra X-ray Observatory, and it has a diameter roughly 200 times the Earth-Sun distance.  Our Sun has a bubble too, called the heliosphere, which protects the planets from cosmic radiation. Also shown in the inset is debris left behind from star formation, observed by Hubble. The debris appears as wings, giving the star its nickname: the Moth.',
  'https://apod.nasa.gov/apod/image/2603/astrosphere_labeled_1024.jpg',
  '')]

In [ ]:
def download_img(image_url, date):
    os.makedirs('images/', exist_ok=True)
    
    filename = f"apod_{date}.jpg"
    filepath = os.path.join('images/', filename)
    
    try:
        img_response = rq.get(image_url)
    
        if img_response.status_code == 200:
            with open(filepath, 'wb') as f:
                f.write(img_response.content)
        
            print(f"image saved: {filepath}")
        else:
            print("An error occurred, the image was not saved, \
            status code:", img_response.status_code)
    
    except Exception as e:
        print("Failed to download image")


In [ ]:
if ".jpg" in data['url']:
    download_img(data['url'], data['date'])

In [53]:
def generate_word():
    max_attempts = 20
    
    for attempt in range(max_attempts):
        
        word_response = rq.get("https://random-words-api.kushcreates.com/api?language=en")
        
        if word_response.status_code != 200:
            continue
            
        try:
            word_data = word_response.json()
            n = randint(0, len(word_data)-1)
            
            random_word = word_data[n]['word']
            
            dict_response = rq.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{random_word}")
            dict_data = dict_response.json()
            
            if isinstance(dict_data, list):
                print(f"Found valid word: {random_word}")
                return dict_data
            else:
                print(f"Word '{random_word}' not in dictionary, trying again...")
                continue
                
        except Exception as e:
            print("Error:", e)
            continue

    # If the word is default, something went wrong with the word generation
    return [{'word': 'default',
             'definition': 'automatic or standard way of acting or responding; \
             go-to or reflex', 'synonyms': ['absence'], 'antonyms': []}]
    

In [62]:
dict_data = generate_word()


Word 'houfs' not in dictionary, trying again...
Word 'kutis' not in dictionary, trying again...
Word 'kales' not in dictionary, trying again...
Found valid word: calix


In [63]:
word_otd = {
    "word": dict_data[0]['word'],
    "definition": dict_data[0]['meanings'][0]['definitions'][0]['definition'],
    "synonyms": ",".join(dict_data[0]['meanings'][0]['definitions'][0]['synonyms']),
    "antonyms": ",".join(dict_data[0]['meanings'][0]['definitions'][0]['antonyms'])
}

In [68]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS words_dict (
        id INTEGER PRIMARY KEY,
        word TEXT UNIQUE NOT NULL,
        definition TEXT UNIQUE NOT NULL,
        synonyms TEXT NOT NULL,
        antonyms TEXT NOT NULL
    )
''')

In [64]:
try:
    cursor.execute(f'''
        INSERT OR IGNORE INTO words_dict 
        (word, definition, synonyms, antonyms)
        VALUES 
        (?, ?, ?, ?)
        ''', (word_otd['word'], word_otd['definition'], word_otd['synonyms'], word_otd['antonyms']))
    
    conn.commit()
except Exception as e:
    print("Error: ", e)


In [69]:
cursor.execute('''
    SELECT * FROM words_dict;
''').fetchall()

[]

In [ ]:
cursor.close()